# Extraction evaluation: the pilot

Step 2 of spec 0011 on `full214780-r4-2610131822-0`: draw and store the 50 pilot
labels (spec 0011.02), then have Claude and Codex extract on them with the
same purpose/risk prompt as Qwen, through the existing `PurposeRiskService`
with their controllers (spec 0012). Each extraction is a new run of 50 rows
under `data/dsld/runs/`, on your subscriptions: 50 calls each.

In [ ]:
from pathlib import Path

import pandas as pd

from engine.controllers.claude_controller import ClaudeController
from engine.controllers.codex_controller import CodexController
from engine.repositories.dsld.runs import loader as runs
from engine.repositories.dsld.runs import paths as run_paths
from engine.repositories.dsld.runs.schema import STRATA
from engine.services.extraction_eval.sample import build_pilot
from engine.services.extraction_eval.schema import Quota
from engine.services.purpose_risk.service import PurposeRiskService

RUN = run_paths.RUNS_ROOT / "full214780-r4-2610131822-0"
SEED = 20261020
LABEL_SET = runs.read_manifest(RUN)["label_set"]

# Starting quotas, spec 0011.02 Quotas table; only pilot and pilot_human are drawn now.
LARGE = {("both", "rare"): 100, ("both", "stitched"): 150, ("both", "verbatim"): 300,
         ("purpose_only", "rare"): 100, ("purpose_only", "stitched"): 150, ("purpose_only", "verbatim"): 150,
         ("risk_only", "rare"): 100, ("risk_only", "stitched"): 150, ("risk_only", "verbatim"): 150,
         ("neither", None): 300}
HUMAN = {s: 30 if s[0] == "neither" else 20 if s[1] == "verbatim" else 10 for s in STRATA}
QUOTAS = {s: Quota(pilot=5, pilot_human=2, large=LARGE[s], human=HUMAN[s]) for s in STRATA}

## Draw and store the pilot

Written once to `RUN/eva/pilot.jsonl` and `RUN/eva/manifests/pilot.json`; the store
refuses to overwrite, so a rerun of this cell reads the stored pilot instead. To
redraw, delete both files by hand.

In [ ]:
if not run_paths.eva_manifest_path(RUN, run_paths.PILOT_MANIFEST_NAME).exists():
    build_pilot(RUN, QUOTAS, SEED)
pilot = runs.read_pilot(RUN)
pilot_ids = [c.label.dsld_id for c in pilot]
pd.DataFrame(
    {"dsld_id": c.label.dsld_id, "group": c.label.stratum[0], "rarest": c.label.stratum[1],
     "rank": c.label.rank, "in_human": c.label.in_human,
     "purpose": len(c.purpose), "risk": len(c.risk)}
    for c in pilot
).groupby(["group", "rarest"], dropna=False).agg(n=("dsld_id", "size"), human=("in_human", "sum"))

## Claude and Codex extract on the pilot

The service prints the new run's path at the end; it is also picked up here as
the run directory that did not exist before. To resume an interrupted one, call
`service.run(run=<that path>)` instead.

In [ ]:
def extract(controller):
    before = set(run_paths.RUNS_ROOT.iterdir())
    with PurposeRiskService(controller=controller) as service:
        service.run(label_set=LABEL_SET, ids=pilot_ids)
    [new] = set(run_paths.RUNS_ROOT.iterdir()) - before
    return new

claude_run = extract(ClaudeController())
claude_run

In [ ]:
codex_run = extract(CodexController())
codex_run

## Side by side

Per pilot label, what Qwen, Claude and Codex extracted. Statuses other than `ok`
are shown, not dropped.

In [ ]:
def by_id(run):
    return {int(r["dsld_id"]): r for r in runs.read_rows(run)}

rows = {"qwen": by_id(RUN), "claude": by_id(claude_run), "codex": by_id(codex_run)}
table = pd.DataFrame(
    {"dsld_id": i, "stratum": f"{c.label.stratum[0]}/{c.label.stratum[1]}",
     **{f"{m}_status": rows[m][i]["status"] for m in rows},
     **{f"{m}_{k}": " | ".join(x["text"] for x in rows[m][i][k]) for m in rows for k in ("purpose", "risk")}}
    for c in pilot for i in [c.label.dsld_id]
)
pd.set_option("display.max_colwidth", 120)
table